# Lab 1: Deep RAG Pipelines for Agentic AI

## What this lab covers
The course touched on RAG briefly. This lab fills the gap — building **production-grade Retrieval-Augmented Generation** pipelines that agents actually use in enterprise systems.

### Why RAG matters for agents
Agents have a fixed context window. RAG is how they access **unlimited, up-to-date knowledge** without hallucinating. Without good RAG, your agent is blind beyond its training cutoff.

### What we'll build
1. **Naive RAG** — baseline: chunk → embed → retrieve → generate
2. **Advanced RAG** — reranking, hybrid search, query expansion
3. **Agentic RAG** — agent decides WHEN and HOW to retrieve
4. **Multi-vector RAG** — summaries + full docs + metadata

---

## Setup

```bash
pip install chromadb langchain-chroma langchain-openai langchain-core langchain sentence-transformers tiktoken langgraph
```

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# Optional: COHERE_API_KEY for reranking
COHERE_API_KEY = os.getenv("COHERE_API_KEY", None)

print("OpenAI key loaded:", bool(OPENAI_API_KEY))

---
## Part 1: Naive RAG — The Baseline

### The 5-step naive RAG pipeline
```
Documents → Chunk → Embed → Store in VectorDB → Retrieve → Generate
```

**Problems with naive RAG (you'll see them):**
- Fixed chunk size loses context
- Semantic search misses keyword matches
- Top-k retrieval returns irrelevant chunks
- No verification that retrieved content is actually relevant

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain.schema import Document

# Sample knowledge base — replace with your own docs or PDFs
SAMPLE_DOCS = [
    """LangGraph is a framework for building stateful, multi-actor applications with LLMs.
    It extends LangChain with the ability to coordinate multiple chains (or actors) across
    multiple steps of computation in a cyclic manner. LangGraph is built on top of LangChain
    and uses its components for the edges in a graph.""",
    
    """CrewAI is a framework for orchestrating role-playing, autonomous AI agents. By fostering
    collaborative intelligence, CrewAI empowers agents to work together seamlessly, tackling
    complex tasks. Each agent has a role, goal, backstory, and set of tools.""",
    
    """The Model Context Protocol (MCP) is an open standard that enables developers to build
    secure, two-way connections between their data sources and AI-powered tools. MCP servers
    expose tools, resources, and prompts that any MCP-compatible client can use.""",
    
    """AutoGen is a framework for building multi-agent systems with message-driven architecture.
    It supports both AgentChat (high-level) and Core (low-level) APIs. The distributed runtime
    uses gRPC for cross-process agent communication.""",
    
    """RAG (Retrieval-Augmented Generation) combines retrieval systems with generative models.
    The retriever fetches relevant documents, and the generator uses them as context to produce
    accurate, grounded responses. This reduces hallucination significantly.""",
]

# Step 1: Create Document objects
docs = [Document(page_content=text, metadata={"source": f"doc_{i}"}) 
        for i, text in enumerate(SAMPLE_DOCS)]

# Step 2: Chunk the documents
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " "]
)
chunks = splitter.split_documents(docs)
print(f"Split {len(docs)} docs into {len(chunks)} chunks")
print(f"\nSample chunk:\n{chunks[0].page_content}")

In [ ]:
# Step 3 & 4: Embed + Store in ChromaDB (local, no server needed)
# NOTE: text-embedding-3-small has an 8191 token limit per chunk (~6000 words).
# Documents longer than this are silently truncated — always chunk first.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="naive_rag_demo",
    persist_directory="./chroma_db"
)
print(f"Stored {vectorstore._collection.count()} chunks in ChromaDB")

In [ ]:
# Step 5: Retrieve + Generate
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def naive_rag(question: str) -> str:
    # Retrieve
    retrieved = retriever.invoke(question)
    context = "\n\n".join([d.page_content for d in retrieved])
    
    # Generate
    response = llm.invoke([
        {"role": "system", "content": "Answer using only the context provided. If the answer isn't in the context, say so."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
    ])
    return response.content

# Test it
q = "How does LangGraph relate to LangChain?"
print(f"Q: {q}")
print(f"A: {naive_rag(q)}")

---
## Part 2: Advanced RAG — Fixing the Problems

### Problem → Solution map:
| Problem | Fix |
|---------|-----|
| Semantic search misses keywords | **Hybrid search** (BM25 + vector) |
| Top-k returns irrelevant chunks | **Reranking** (cross-encoder) |
| Vague queries return bad results | **Query expansion / HyDE** |
| Chunks lose surrounding context | **Parent-child chunking** |

In [ ]:
# --- Fix 1: Query Expansion ---
# Generate multiple versions of the query to improve recall

from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel
from typing import List
import json

class QueryExpansion(BaseModel):
    queries: List[str]

def expand_query(question: str, n: int = 3) -> List[str]:
    """Generate n alternative phrasings of the question."""
    response = llm.invoke([
        {"role": "system", "content": f"""Generate {n} different ways to ask the same question.
Return ONLY a JSON object like: {{"queries": ["q1", "q2", "q3"]}}"""},
        {"role": "user", "content": question}
    ])
    try:
        parsed = json.loads(response.content)
        return [question] + parsed["queries"]  # include original
    except (json.JSONDecodeError, KeyError):
        # LLM returned malformed JSON — fall back to the original question only
        print(f"[expand_query] Could not parse LLM response, using original query only.")
        return [question]

original_q = "What makes LangGraph different from regular LangChain?"
expanded = expand_query(original_q)
print("Original:", original_q)
print("Expanded queries:")
for i, q in enumerate(expanded, 1):
    print(f"  {i}. {q}")

In [ ]:
# --- Fix 2: Multi-query retrieval with deduplication ---

def multi_query_retrieve(question: str, k: int = 3) -> List[Document]:
    """Retrieve docs using multiple query phrasings, deduplicate by content."""
    queries = expand_query(question, n=2)  # 3 queries total
    
    seen_contents = set()
    all_docs = []
    
    for q in queries:
        retrieved = retriever.invoke(q)
        for doc in retrieved:
            content_hash = hash(doc.page_content)
            if content_hash not in seen_contents:
                seen_contents.add(content_hash)
                all_docs.append(doc)
    
    print(f"Retrieved {len(all_docs)} unique chunks across {len(queries)} queries")
    return all_docs[:k * 2]  # return top N unique

docs_retrieved = multi_query_retrieve("How does LangGraph handle state?")
for i, d in enumerate(docs_retrieved, 1):
    print(f"\nChunk {i}: {d.page_content[:100]}...")

In [ ]:
# --- Fix 3: Reranking (without paid Cohere — using a simple relevance scorer) ---
# In production you'd use: cohere.rerank() or a cross-encoder model

def rerank_with_llm(question: str, docs: List[Document], top_n: int = 3) -> List[Document]:
    """Use LLM to score and rerank retrieved documents by relevance."""
    import json
    
    docs_text = "\n\n".join([
        f"[{i}] {doc.page_content}" for i, doc in enumerate(docs)
    ])
    
    response = llm.invoke([{
        "role": "user",
        "content": f"""Score each document's relevance to the question on a scale 0-10.
Return ONLY JSON: {{"scores": [score0, score1, ...]}}

Question: {question}

Documents:
{docs_text}"""
    }])
    
    scores = json.loads(response.content)["scores"]
    ranked = sorted(zip(scores, docs), reverse=True)
    return [doc for _, doc in ranked[:top_n]]

question = "What is the difference between LangGraph and AutoGen?"
initial_docs = multi_query_retrieve(question, k=3)
reranked_docs = rerank_with_llm(question, initial_docs)

print("Top reranked chunks:")
for i, doc in enumerate(reranked_docs, 1):
    print(f"  {i}. {doc.page_content[:120]}...")

In [ ]:
# --- Advanced RAG Pipeline (combined) ---

def advanced_rag(question: str) -> str:
    """Full advanced RAG: expand → retrieve → rerank → generate."""
    # 1. Multi-query retrieval
    candidates = multi_query_retrieve(question, k=4)
    
    # 2. Rerank
    top_docs = rerank_with_llm(question, candidates, top_n=3)
    
    # 3. Generate with grounded context
    context = "\n\n".join([d.page_content for d in top_docs])
    response = llm.invoke([
        {"role": "system", "content": """Answer using only the context. 
If the answer isn't there, say 'I don't have enough information.'
Cite which part of the context supports your answer."""},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
    ])
    return response.content

print(advanced_rag("How do agent frameworks handle multi-agent communication?"))

---
## Part 3: Agentic RAG — The Agent Decides When to Retrieve

### Key insight
In naive RAG, you **always** retrieve. In agentic RAG, the agent:
1. Decides **whether** retrieval is needed
2. Decides **what** to retrieve (query formulation)
3. Decides **whether the results are good enough** (self-evaluation)
4. Can **retry with a different query** if results are poor

This is the **Corrective RAG (CRAG)** pattern — widely used in production.

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional
from openai import OpenAI
import json

client = OpenAI()

# --- State definition ---
class RAGState(TypedDict):
    question: str
    retrieved_docs: List[Document]
    retrieval_score: float        # how relevant are the retrieved docs (0-1)
    answer: Optional[str]
    retry_count: int

# --- Nodes ---

def retrieve_node(state: RAGState) -> RAGState:
    """Retrieve documents for the current question."""
    question = state["question"]
    docs = multi_query_retrieve(question, k=3)
    # NOTE: retrieved_docs is intentionally reset here on every retry pass.
    # We keep only the LATEST retrieval attempt's docs — not cumulative across
    # retries — because the rephrased query is expected to return better results.
    # If you want to accumulate docs across retries, merge with state["retrieved_docs"].
    return {**state, "retrieved_docs": docs}

def grade_retrieval_node(state: RAGState) -> RAGState:
    """Score the relevance of retrieved docs. If poor, we'll rephrase and retry."""
    question = state["question"]
    docs = state["retrieved_docs"]
    
    if not docs:
        return {**state, "retrieval_score": 0.0}
    
    context = "\n".join([d.page_content for d in docs[:2]])
    response = llm.invoke([{
        "role": "user",
        "content": f"""Rate how relevant these documents are to the question (0.0 to 1.0).
Return ONLY a JSON: {{"score": 0.85}}

Question: {question}
Documents: {context}"""
    }])
    try:
        score = json.loads(response.content)["score"]
    except (json.JSONDecodeError, KeyError):
        score = 0.0
    print(f"Retrieval quality score: {score}")
    return {**state, "retrieval_score": score}

def rephrase_query_node(state: RAGState) -> RAGState:
    """If retrieval was poor, rephrase the question and retry."""
    response = llm.invoke([{
        "role": "user",
        "content": f"Rephrase this question to be more specific and retrievable: {state['question']}"
    }])
    new_question = response.content.strip()
    print(f"Rephrased to: {new_question}")
    return {**state, "question": new_question, "retry_count": state["retry_count"] + 1}

def generate_node(state: RAGState) -> RAGState:
    """Generate the final answer."""
    context = "\n\n".join([d.page_content for d in state["retrieved_docs"]])
    response = llm.invoke([
        {"role": "system", "content": "Answer clearly using the context. Be concise."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {state['question']}"}
    ])
    return {**state, "answer": response.content}

# --- Routing logic ---
# Threshold of 0.5 is a reasonable starting point for this demo corpus.
# In production, calibrate it against your eval dataset (Lab 2): run 50+
# retrieval tests, compute precision/recall at different thresholds, and
# pick the value that minimises false negatives (missed good results) while
# keeping retry count below 2 per request on average.
def route_after_grading(state: RAGState) -> str:
    if state["retrieval_score"] >= 0.5:
        return "generate"       # good enough → generate
    elif state["retry_count"] < 2:
        return "rephrase"       # try a better query
    else:
        return "generate"       # give up retrying, generate with what we have

# --- Build the graph ---
graph = StateGraph(RAGState)
graph.add_node("retrieve", retrieve_node)
graph.add_node("grade", grade_retrieval_node)
graph.add_node("rephrase", rephrase_query_node)
graph.add_node("generate", generate_node)

graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "grade")
graph.add_conditional_edges("grade", route_after_grading, {
    "generate": "generate",
    "rephrase": "rephrase"
})
graph.add_edge("rephrase", "retrieve")  # loop back
graph.add_edge("generate", END)

crag = graph.compile()
print("Corrective RAG graph compiled.")

In [ ]:
# Run the Agentic RAG pipeline
result = crag.invoke({
    "question": "How do I handle memory in multi-agent systems?",
    "retrieved_docs": [],
    "retrieval_score": 0.0,
    "answer": None,
    "retry_count": 0
})

print("\n" + "="*50)
print("FINAL ANSWER:")
print(result["answer"])

---
## Part 4: Chunking Strategies Deep Dive

Chunking is the most underrated part of RAG. Bad chunking = bad retrieval = bad answers.

In [ ]:
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    TokenTextSplitter
)

SAMPLE_MARKDOWN = """
# Agent Frameworks

## LangGraph
LangGraph uses a StateGraph to define workflows. Each node is a Python function.
Edges define transitions. Conditional edges enable routing logic.

## CrewAI
CrewAI uses YAML-defined agents and tasks. Crews orchestrate agents in sequences
or hierarchies. Memory is built in.

## AutoGen
AutoGen uses message-passing between agents. The Core API gives full control.
The AgentChat API is higher-level for rapid development.
"""

# Strategy 1: Fixed-size character chunks
char_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
char_chunks = char_splitter.create_documents([SAMPLE_MARKDOWN])
print(f"Strategy 1 (char): {len(char_chunks)} chunks")

# Strategy 2: Markdown-aware (preserves headers as metadata)
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[
    ("#", "section"),
    ("##", "subsection")
])
md_chunks = md_splitter.split_text(SAMPLE_MARKDOWN)
print(f"\nStrategy 2 (markdown): {len(md_chunks)} chunks")
for c in md_chunks:
    print(f"  Metadata: {c.metadata} | Content: {c.page_content[:60]}...")

# Strategy 3: Token-based (respects model token limits precisely)
token_splitter = TokenTextSplitter(chunk_size=50, chunk_overlap=10)
token_chunks = token_splitter.create_documents([SAMPLE_MARKDOWN])
print(f"\nStrategy 3 (token): {len(token_chunks)} chunks")

In [ ]:
# --- Parent-Child chunking (best of both worlds) ---
# Small chunks for precise retrieval, large parent for full context

from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=0)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)

# Storage for parent documents
docstore = InMemoryStore()

parent_child_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# Add documents (child chunks go to vectorstore, parents to docstore)
parent_child_retriever.add_documents(docs)

# Retrieve: finds small child chunks, returns large parent chunks
results = parent_child_retriever.invoke("How does LangGraph work?")
print(f"Retrieved {len(results)} parent chunks")
for r in results:
    print(f"Parent chunk ({len(r.page_content)} chars): {r.page_content[:150]}...")

# --- Compare parent-child vs naive retrieval ---
# This shows the concrete improvement parent-child chunking gives you.
naive_results = retriever.invoke("How does LangGraph work?")
print(f"\n--- Comparison: Naive vs Parent-Child ---")
print(f"Naive retrieval returned {len(naive_results)} chunks, avg size: "
      f"{sum(len(d.page_content) for d in naive_results) // max(len(naive_results),1)} chars")
print(f"Parent-child returned {len(results)} chunks, avg size: "
      f"{sum(len(d.page_content) for d in results) // max(len(results),1)} chars")
print("\nNaive top chunk:")
print(f"  {naive_results[0].page_content[:200] if naive_results else 'none'}...")
print("\nParent-child top chunk (has full surrounding context):")
print(f"  {results[0].page_content[:200] if results else 'none'}...")
print("\nTakeaway: parent-child returns larger, context-rich chunks while still "
      "using small chunks for precise matching — fewer out-of-context snippets.")

---
## Summary: RAG Quality Checklist

Before shipping a RAG system, verify:

| Check | How to verify |
|-------|---------------|
| Chunk size appropriate | Does each chunk contain a complete thought? |
| Embedding model fits domain | Run retrieval tests on 20+ real questions |
| Retrieval recall is high | Are ground-truth docs appearing in top-k? |
| Reranking improves precision | Does reranked order match human judgment? |
| Generation is grounded | Does the answer only reference retrieved context? |
| Fallback handles out-of-scope | Does it say 'I don't know' vs hallucinate? |

## Next Steps
- Lab 2 covers how to **evaluate** your RAG pipeline systematically
- Try swapping ChromaDB for **Pinecone** or **Weaviate** for production scale
- For domain-specific RAG, consider **fine-tuning the embedding model** on your corpus